In [1]:
import neo4j

import pandas as pd

from IPython.display import display

import psycopg2

In [2]:
driver = neo4j.GraphDatabase.driver(uri="neo4j://neo4j:7687", auth=("neo4j","ucb_mids_w205"))

In [3]:
session = driver.session(database="neo4j")

my_neo4j_wipe_out_database() - since community edition can only have 1 database "neo4j", this function will wipe out all the nodes and relationships

In [4]:
#def my_neo4j_wipe_out_database():
    #"wipe out database by deleting all nodes and relationships"
    
   # query = "match (node)-[relationship]->() delete node, relationship"
    #session.run(query)
    
   # query = "match (node) delete node"
   # session.run(query)

my_neo4j_run_query_pandas() will run a Cypher query and put the results in a Pandas dataframe; easy to see how you can use Python to manipulate the returned data

In [5]:
def my_neo4j_run_query_pandas(query, **kwargs):
    "run a query and return the results in a pandas dataframe"
    
    result = session.run(query, **kwargs)
    
    df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    
    return df

my_neo4j_nodes_relationships() will print the nodes (assumes a name property) and relationships

In [6]:
def my_neo4j_nodes_relationships():
    "print all the nodes and relationships"
   
    print("-------------------------")
    print("  Nodes:")
    print("-------------------------")
    
    query = """
        match (n) 
        return n.name as node_name, labels(n) as labels
        order by n.name
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_nodes = df.shape[0]
    
    display(df)
    
    print("-------------------------")
    print("  Relationships:")
    print("-------------------------")
    
    query = """
        match (n1)-[r]->(n2) 
        return n1.name as node_name_1, labels(n1) as node_1_labels, 
            type(r) as relationship_type, n2.name as node_name_2, labels(n2) as node_2_labels
        order by node_name_1, node_name_2
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_relationships = df.shape[0]
    
    display(df)
    
    density = (2 * number_relationships) / (number_nodes * (number_nodes - 1))
    
    print("-------------------------")
    print("  Density:", f'{density:.1f}')
    print("-------------------------")

In [7]:
def my_neo4j_create_node(station_name):
    "create a node with label Station"
    
    query = """
    
    CREATE (:Station {name: $station_name})
    
    """
    
    session.run(query, station_name=station_name)
    

In [8]:
def my_neo4j_create_relationship_one_way(from_station, to_station, weight):
    "create a relationship one way between two stations with a weight"
    
    query = """
    
    MATCH (from:Station), 
          (to:Station)
    WHERE from.name = $from_station and to.name = $to_station
    CREATE (from)-[:LINK {weight: $weight}]->(to)
    
    """
    
    session.run(query, from_station=from_station, to_station=to_station, weight=weight)

In [9]:
def my_neo4j_create_relationship_two_way(from_station, to_station, weight):
    "create relationships two way between two stations with a weight"
    
    query = """
    
    MATCH (from:Station), 
          (to:Station)
    WHERE from.name = $from_station and to.name = $to_station
    CREATE (from)-[:LINK {weight: $weight}]->(to),
           (to)-[:LINK {weight: $weight}]->(from)
    
    """
    
    session.run(query, from_station=from_station, to_station=to_station, weight=weight)
    

In [10]:
def my_neo4j_shortest_path(from_station, to_station):
    "given a from station and to station, run and print the shortest path"
    
    query = "CALL gds.graph.drop('ds_graph', false)"
    session.run(query)

    query = "CALL gds.graph.project('ds_graph', 'Station', 'LINK', {relationshipProperties: 'weight'})"
    session.run(query)

    query = """

    MATCH (source:Station {name: $source}), (target:Station {name: $target})
    CALL gds.shortestPath.dijkstra.stream(
        'ds_graph', 
        { sourceNode: source, 
          targetNode: target, 
          relationshipWeightProperty: 'weight'
        }
    )
    YIELD index, sourceNode, targetNode, totalCost, nodeIds, costs, path
    RETURN
        gds.util.asNode(sourceNode).name AS from,
        gds.util.asNode(targetNode).name AS to,
        totalCost,
        [nodeId IN nodeIds | gds.util.asNode(nodeId).name] AS nodes,
        costs
    ORDER BY index

    """

    result = session.run(query, source=from_station, target=to_station)
    
    for r in result:
        
        total_cost = int(r['totalCost'])
        
        print("\n--------------------------------")
        print("   Total Cost: ", total_cost)
        print("   Minutes: ", round(total_cost / 60.0,1))
        print("--------------------------------")
        
        nodes = r['nodes']
        costs = r['costs']
        
        i = 0
        previous = 0
        
        for n in nodes:
            
            print(n + ", " + str(int(costs[i]) - previous)  + ", " + str(int(costs[i])))
            
            previous = int(costs[i])
            i += 1

In [11]:
def my_calculate_box(point, miles):
    "Given a point and miles, calculate the box in form left, right, top, bottom"
    
    geod = Geodesic.WGS84

    kilometers = miles * 1.60934
    meters = kilometers * 1000

    g = geod.Direct(point[0], point[1], 270, meters)
    left = (g['lat2'], g['lon2'])

    g = geod.Direct(point[0], point[1], 90, meters)
    right = (g['lat2'], g['lon2'])

    g = geod.Direct(point[0], point[1], 0, meters)
    top = (g['lat2'], g['lon2'])

    g = geod.Direct(point[0], point[1], 180, meters)
    bottom = (g['lat2'], g['lon2'])
    
    return(left, right, top, bottom)

In [12]:
def my_station_get_zips(station, miles):
    "given a station, pull all zip codes with miles distance, print them, sum the population"
    
    connection.rollback()
    
    query = "select latitude, longitude from stations "
    query += "where station = '" + station + "'"
    
    cursor.execute(query)
    
    connection.rollback()
    
    rows = cursor.fetchall()
    
    for row in rows:
        latitude = row[0]
        longitude = row[1]
        
    point = (latitude, longitude)
        
    (left, right, top, bottom) = my_calculate_box(point, miles)
    
    query = "select zip, population from zip_codes "
    query += " where latitude >= " + str(bottom[0])
    query += " and latitude <= " + str(top [0])
    query += " and longitude >= " + str(left[1])
    query += " and longitude <= " + str(right[1])
    query += " order by 1 "

    cursor.execute(query)
    
    connection.rollback()
    
    rows = cursor.fetchall()
    
    print("\n-------------------------------------------------------------------------------")
    print("  Zip Codes within " + str(miles) + " mile(s) of " + station + " BART Station")
    print("-------------------------------------------------------------------------------\n")
    
    total_population = 0
    
    for row in rows:
        zip = row[0]
        population = row[1]
        print("     zip:", zip, "  population: ", f'{population:10,}')
        total_population += population
        
    
    print("\n-------------------------------------------------------------------------------")
    print("  Total Population: ", f'{total_population:10,}')
    print("-------------------------------------------------------------------------------")

In [13]:
connection = psycopg2.connect(
    user = "postgres",
    password = "ucb",
    host = "postgres",
    port = "5432",
    database = "postgres"
)

In [14]:
cursor = connection.cursor()

### Page Rank
- higher scores are the more significant hubs

In [22]:
query = "CALL gds.graph.drop('ds_graph', false)"
session.run(query)

query = "CALL gds.graph.project('ds_graph', 'Station', 'LINK', {relationshipProperties: 'weight'})"
session.run(query)

pagerank_query = """
CALL gds.pageRank.stream('ds_graph')
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS station, score
ORDER BY score DESC
"""

my_neo4j_run_query_pandas(pagerank_query)

,station,score
0,blue Coliseum,0.689786
1,orange Coliseum,0.689174
2,green Coliseum,0.688970
3,blue Bay Fair,0.675695
4,yellow MacArthur,0.675481
...,...,...
209,depart Union City,0.150000
210,depart Walnut Creek,0.150000
211,depart Warm Springs,0.150000
212,depart West Dublin,0.150000


### Closeness centrality
- stations with higher scores indicate that it is more centrally located in the network, implying shorter paths on average to all other stations

In [15]:
query = "CALL gds.graph.drop('ds_graph', false)"
session.run(query)

query = "CALL gds.graph.project('ds_graph', 'Station', 'LINK', {relationshipProperties: 'weight'})"
session.run(query)

closeness_query = """
CALL gds.beta.closeness.stream('ds_graph')
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS name, score as closeness
ORDER BY score DESC
"""

my_neo4j_run_query_pandas(closeness_query)

,name,closeness
0,yellow West Oakland,0.138488
1,green West Oakland,0.137902
2,red West Oakland,0.137553
3,blue West Oakland,0.136975
4,yellow 12th Street,0.135607
...,...,...
209,depart Union City,0.000000
210,depart Walnut Creek,0.000000
211,depart Warm Springs,0.000000
212,depart West Dublin,0.000000


### Betweenness 
- higher points indicate 

In [21]:
query = "CALL gds.graph.drop('ds_graph', false)"
session.run(query)

query = "CALL gds.graph.project('ds_graph', 'Station', 'LINK', {relationshipProperties: 'weight'})"
session.run(query)

betweenness_query = """
CALL gds.betweenness.stream('ds_graph')
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS name, score AS betweenness
ORDER BY score DESC
"""

my_neo4j_run_query_pandas(betweenness_query)


,name,betweenness
0,yellow MacArthur,5999.809223
1,yellow Rockridge,5509.000000
2,orange Lake Merritt,5155.831877
3,orange 12th Street,5139.715461
4,yellow Orinda,4997.000000
...,...,...
209,arrive Warm Springs,0.000000
210,depart West Dublin,0.000000
211,arrive West Dublin,0.000000
212,depart West Oakland,0.000000


### Shortest path on all pairs
- showing all the distance between all pairs

In [23]:
query = "CALL gds.graph.drop('ds_graph', false)"
session.run(query)

query = "CALL gds.graph.project('ds_graph', 'Station', 'LINK', {relationshipProperties: 'weight'})"
session.run(query)


shortest_path_query = """
CALL gds.allShortestPaths.stream('ds_graph', {relationshipWeightProperty: 'weight'})
YIELD sourceNodeId, targetNodeId, distance
RETURN gds.util.asNode(sourceNodeId).name AS source, 
       gds.util.asNode(targetNodeId).name AS target, 
       distance
"""

my_neo4j_run_query_pandas(shortest_path_query)


,source,target,distance
0,yellow Antioch,yellow Antioch,0.0
1,yellow Antioch,orange Ashby,3479.0
2,yellow Antioch,red Ashby,3479.0
3,yellow Antioch,blue Balboa Park,5088.0
4,yellow Antioch,green Balboa Park,5088.0
...,...,...,...
26991,red 19th Street,yellow 19th Street,67.0
26992,red 19th Street,blue 24th Street Mission,1657.0
26993,red 19th Street,green 24th Street Mission,1657.0
26994,red 19th Street,red 24th Street Mission,1380.0


### Jaccard Similarity
- showing how many of the same stations between two stations and give similarity ranking

In [25]:
query = "CALL gds.graph.drop('ds_graph', false)"
session.run(query)

query = "CALL gds.graph.project('ds_graph', 'Station', 'LINK', {relationshipProperties: 'weight'})"
session.run(query)

jaccard_query = """
CALL gds.nodeSimilarity.stream('ds_graph')
YIELD node1, node2, similarity
RETURN gds.util.asNode(node1).name AS station1, 
       gds.util.asNode(node2).name AS station2, 
       similarity
"""

my_neo4j_run_query_pandas(jaccard_query)


,station1,station2,similarity
0,yellow Antioch,depart Pittsburg Center,0.500000
1,yellow Antioch,yellow Pittsburg,0.250000
2,orange Ashby,red Downtown Berkeley,0.333333
3,orange Ashby,red MacArthur,0.285714
4,orange Ashby,depart Downtown Berkeley,0.200000
...,...,...,...
1259,yellow 24th Street Mission,red Glen Park,0.200000
1260,yellow 24th Street Mission,blue 16th Street Mission,0.200000
1261,yellow 24th Street Mission,red 16th Street Mission,0.200000
1262,yellow 24th Street Mission,green 16th Street Mission,0.200000
